In [1]:
import pandas as pd
import re
import requests
import time
import json

In [2]:
# Read in the main resale dataset
df = pd.read_csv("Resale flat prices based on registration date from Jan-2017 onwards.csv")

In [3]:
# Keep only block and street_name
df = df[["block", "street_name"]]

# Create a full location column by combining block and street name
df["Location"] = df["block"] + " " + df["street_name"]

# Save to csv file
df.to_csv("location_combined.csv", index=False)

## Extract Coordinates Using OneMap API 

In [4]:
# === CONFIGURATION ===
INPUT_FILE = 'location_combined.csv'
OUTPUT_FILE = 'location_with_latlong.csv'

In [5]:
# === ONEMAP QUERY FUNCTION ===
def get_coords_onemap(address):
    for _ in range(3):   # try at most 3 times
        try:
            time.sleep(0.3)
            url = "https://www.onemap.gov.sg/api/common/elastic/search"
            params = {
                "searchVal": address,
                "returnGeom": "Y",
                "getAddrDetails": "Y",
                "pageNum": 1
            }
            response = requests.get(url, params=params, timeout=60)

            data = response.json()
            if data["found"] > 0:
                result = data["results"][0]
                return f"{result['LATITUDE']}, {result['LONGITUDE']}"
        except Exception as e:
            print("Error:", address, e)
        return "NIL"

df = pd.read_csv(INPUT_FILE)

df['Lat Long'] = df['Location'].apply(get_coords_onemap)

df.to_csv(OUTPUT_FILE, index=False)
print("Done!")

Done!
